In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import os
import torch
import pickle
import config

from sentence_transformers import SentenceTransformer,InputExample,losses

from torch.utils.data import DataLoader

from src.metric import *


/tmp/ipykernel_5833/4205643316.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer,InputExample,losses


In [3]:
torch.set_float32_matmul_precision("high")

In [4]:
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [5]:
path=config.CLEANED_DATA_DIR

In [6]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [7]:
bi_encoder= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=config.device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
BATCH_SIZE=config.BASELINE_BATCH_SIZE

In [9]:
g = torch.Generator()
g.manual_seed(config.SEED)

In [10]:
bi_train_examples=[
    InputExample(texts=[j,r],label=float(config.label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

bi_train_dataloader=DataLoader(bi_train_examples,shuffle=True,batch_size=BATCH_SIZE)

bi_train_loss=losses.CoSENTLoss(bi_encoder)

In [11]:
val_resume_emb = bi_encoder.encode(val_df['resume_text'].tolist(),
                              batch_size=BATCH_SIZE,convert_to_tensor=True,
                              show_progress_bar=True)
val_jd_emb = bi_encoder.encode(val_df['job_description_text'].tolist(),
                              batch_size=BATCH_SIZE,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(val_resume_emb,val_jd_emb).cpu().numpy()

metrics=model_evaluation(scores,val_df,'job_description_text')
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Spearman: 0.33707315568770985
Top-3 Accuracy: 1.0
NDCG: 0.6698497775737843
MAP: 0.763996419409418
MRR: 0.8370039682539682


In [12]:
model_save_path=os.path.join(config.BASELINE_MODEL_DIR,'bi_encoder_baseline')
os.makedirs(config.BASELINE_MODEL_DIR,exist_ok=True)

In [13]:
epochs = 4
best_score = float('-inf')
min_delta=0.01

print("\tTraining Phase")
for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}----------")

    bi_encoder.fit(
        train_objectives=[(bi_train_dataloader, bi_train_loss)],
        epochs=1,
        warmup_steps=int(len(bi_train_dataloader) * epochs * 0.1),
        show_progress_bar=True
    )

    val_resume_emb=bi_encoder.encode(
        val_df['resume_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )
    val_jd_emb =bi_encoder.encode(
        val_df['job_description_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )

    scores = torch.cosine_similarity(val_resume_emb, val_jd_emb).cpu().numpy()

    metrics = model_evaluation(scores, val_df, 'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])

    final_score = 0.6*metrics['ndcg_val']+0.3*metrics['map_score']+0.1*metrics['mrr_score']

    if final_score>best_score+min_delta:
        best_score=final_score
        bi_encoder.save(model_save_path)
        
    


	Training Phase
Epoch: 1----------


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


NDCG: 0.6887492606401059
MAP: 0.7806188615372163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

NDCG: 0.765076876209396
MAP: 0.8422521559279733


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 4----------


Step,Training Loss


NDCG: 0.7758380668407097
MAP: 0.8506450379840199


In [14]:
bi_encoder= SentenceTransformer(model_save_path,device=config.device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
val_resume_emb = bi_encoder.encode(val_df['resume_text'].tolist(),
                              batch_size=BATCH_SIZE,convert_to_tensor=True,
                              show_progress_bar=True)
val_jd_emb = bi_encoder.encode(val_df['job_description_text'].tolist(),
                              batch_size=BATCH_SIZE,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(val_resume_emb,val_jd_emb).cpu().numpy()

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [16]:
metrics=model_evaluation(scores,val_df,'job_description_text')
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.5572673976794694
Top-3 Accuracy: 1.0
NDCG: 0.765076876209396
MAP: 0.8422521559279733
MRR: 0.8814484126984127


In [17]:
eval_df=val_df.copy()
eval_df['score']=scores
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 0.225
% groups with spread < 0.1:0.074


In [18]:
false_neg = eval_df[(eval_df['label'] == 2) & (eval_df['score'] < -0.3)]

print(f"Good Fit resumes scoring below -0.3: {len(false_neg)}")
print("\nSample false_neg resumes:")

i=0
for jd,group in false_neg.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

Good Fit resumes scoring below -0.3: 0

Sample false_neg resumes:


In [20]:
confused = eval_df[(eval_df['label'] == 1) & (eval_df['score'] < 1.2) & (eval_df['score'] > -0.1)]

print(f"confused predictions: {len(confused)}")
print("\nSample confused resumes:")

i=0
for jd,group in confused.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

confused predictions: 285

Sample confused resumes:
----------------------------------------------------------------------------------------------------
JD:  experienced in salesforce industries communications cloud.certification a plus 10+ years in sfdc 5+ years of experience in telecom domain solutioning for quoteto cash architect software solutions usi
Score: 0.832
Resume: Professional SummaryBusiness Intelligence Consultant with a 10-year career in data warehousing, business intelligence reporting, and data management architecture. Progressive developer and technical team lead with a strength in design & development, as well as driving performance, reducing inefficie

Score: 0.798
Resume: SummaryExperienced Data Analyst who responds to shifting business needs and priorities in a systematic and effective way.  Excels at implementing operational assessments and conducting functional requirements analysis for businesses of all sized.  Committed to maintaining cutting edge technical sk

In [21]:
test_resume_emb=bi_encoder.encode(test_df['resume_text'].tolist(),
                              batch_size=BATCH_SIZE,convert_to_tensor=True,
                              show_progress_bar=True)
test_jd_emb = bi_encoder.encode(test_df['job_description_text'].tolist(),
                              batch_size=BATCH_SIZE,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(test_resume_emb,test_jd_emb).cpu().numpy()

metrics=model_evaluation(scores,test_df,'job_description_text')

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [22]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.3918299704926086
Top-3 Accuracy: 1.0
NDCG: 0.6470203998041301
MAP: 0.7479806457300369
MRR: 0.7726971116315378
